In [1]:
import json
import requests

## 1. Define the Search Tool

In [84]:
def search_wikipedia(query: str) -> dict:
    """
    Search Wikipedia for a given query string using the MediaWiki API.

    This function sends a request to the Wikipedia search endpoint
    (`action=query&list=search`) and returns the parsed JSON response
    containing a list of matching pages.

    Args:
        query (str): The search term to query on Wikipedia
                     (e.g., "lesser capybara").

    Returns:
        dict: A dictionary representation of the JSON response from the
              Wikipedia API. The search results can be accessed via:
              `data['query']['search']`, which is a list of result objects.
              Each result typically contains fields like 'title', 'snippet',
              and 'pageid'.

    Notes:
        - The API returns a limited number of results per request (default: 10).
    """
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
    YOUR_QUERY = query
    user_query = f'https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={YOUR_QUERY}'
    response = requests.get(user_query, headers=headers)
    data = response.json()
    return data

In [86]:
query = 'capybara'
data = search_wikipedia(query)

print('response user query:', '\n')

print(data)
print('\n', 40 * '=', '\n')
# print('Page title:', data['query']['search'][0]['title'], '\n')

search_results = data.get("query", {}).get("search", [])

if search_results:
    page_title = search_results[0]["title"]
    print('Page title:', page_title)
else:
    print("No results found")

print('\n', 40 * '=', '\n')
result_count = len(data["query"]["search"])
print('Num results:', result_count)

response user query: 

{'batchcomplete': '', 'continue': {'sroffset': 10, 'continue': '-||'}, 'query': {'searchinfo': {'totalhits': 909, 'suggestion': 'capivara', 'suggestionsnippet': 'capivara'}, 'search': [{'ns': 0, 'title': 'Capybara', 'pageid': 6776, 'size': 37074, 'wordcount': 3734, 'snippet': 'The <span class="searchmatch">capybara</span> or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South America except Chile. It is', 'timestamp': '2026-04-17T08:54:32Z'}, {'ns': 0, 'title': 'Lesser capybara', 'pageid': 23188846, 'size': 5578, 'wordcount': 581, 'snippet': 'The lesser <span class="searchmatch">capybara</span> (Hydrochoerus isthmius) is a large semi-aquatic rodent found in South America that has vast similarities with, yet subtle differences', 'timestamp': '2026-03-23T22:32:11Z'}, {'ns': 0, 'title': 'Capybara (disambiguation)', 'pageid': 69085306, 'size': 516, 'wordcount': 100, 'snippet': 'L

In [87]:
def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia for a given query string and return the title of the
    first matching page.

    This function uses the MediaWiki API (`action=query&list=search`)
    to find relevant Wikipedia pages. It extracts and returns only the
    title of the top search result, which can be used as input for other
    tools (e.g., to fetch the full page content).

    Args:
        query (str): The search term to look up on Wikipedia
                     (e.g., "capybara", "machine learning").

    Returns:
        str: The title of the first search result. If no results are found,
             returns the string "No results found.".
    """
    
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0"}
    url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
    
    response = requests.get(url, headers=headers)
    data = response.json()
    
    results = data.get("query", {}).get("search", [])
    
    if not results:
        return "No results found."
    
    return results[0]["title"]

In [88]:
query = 'lesser+capybara'
search_wikipedia(query)

'Lesser capybara'

## 2. Analyzing Search Results

In [89]:
query = 'capybara'
data = search_wikipedia(query)

count_capybara = 0

for r in search_results:
    # print(r['title'].lower())
    if 'capybara' in r['title'].lower():
        count_capybara += 1
print('count capybara:', count_capybara)

count capybara: 5


## 3. Define the Get Page Tool

In [90]:
def get_page(page_title: str) -> str:
    """
    Fetch the raw Wikitext content of a Wikipedia page by its title.

    This function sends a request to the Wikipedia endpoint with
    `action=raw` and returns the full page content in Wikitext format.

    Args:
        page_title (str): The title of the Wikipedia page to retrieve
                          (e.g., "Capybara").

    Returns:
        str: The raw Wikitext content of the requested Wikipedia page.
             This is a plain string and can be processed or summarized
             by downstream components.
    """
    
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
    url = f'https://en.wikipedia.org/w/index.php?title={page_title}&action=raw'
    response = requests.get(url, headers=headers)
    
    return response.text

In [91]:
response = get_page('capybara')

In [92]:
len(response)

36946

## 4. Agent Setup

In [42]:
from pydantic_ai import Agent

In [77]:
tools = [search_wikipedia, get_page]

instructions = """
You are a research assistant.

Workflow:
1. Search Wikipedia for the topic
2. Get the page content
3. Answer the users question

Always use the tools. Do not hallucinate content.
"""

In [78]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [79]:
query = "I'm looking for infomration about capybaras."
result = await search_agent.run(query)
print(result.output)

The **capybara** (scientific name: *Hydrochoerus hydrochaeris*) is the largest living rodent, native to South America, where it can be found in every country except Chile. These semi-aquatic herbivores inhabit savannas and dense forests, residing close to water bodies like lakes and rivers. Capybaras mainly feed on grasses and aquatic plants.

### Key Characteristics:
- **Size**: Adults typically measure between 106 to 134 cm in length and weigh between 35 to 66 kg, with females generally being slightly heavier.
- **Appearance**: They have a barrel-shaped body covered in reddish-brown fur that fades to yellowish-brown underneath. Capybaras have slightly webbed feet, contributing to their swimming abilities.
- **Social Behavior**: Capybaras are highly social animals, often found in groups of 10-20, but sometimes up to 100. They communicate using barks and whistles.

### Ecology:
Capybaras are proficient swimmers, capable of holding their breath for up to five minutes. They are also herb

## 5. Testing Your Agent - Single Page

In [80]:
query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"
result = await search_agent.run(query)
print(result.output)

The Wikipedia page for the **capybara** discusses various aspects of this animal, which is recognized as the largest living rodent. Here are key points from the page:

1. **Species Information**: The capybara (scientific name *Hydrochoerus hydrochaeris*) is native to South America, where it inhabits savannas and forests, often near bodies of water. 

2. **Physical Description**: Capybaras have a distinct barrel-shaped body, reddish-brown fur, and they typically weigh between 35 to 66 kg (about 77 to 146 lbs).

3. **Social Structure**: They are highly social animals, often found in groups ranging from 10 to 20 individuals, but can congregate in larger groups during certain seasons.

4. **Diet**: Capybaras are herbivores, primarily feeding on grasses and aquatic plants. They demonstrate unique feeding behavior, including coprophagy (eating their own feces).

5. **Ecology and Behavior**: They are excellent swimmers and can hold their breath underwater for up to five minutes. The capybara 

In [81]:
messages = result.all_messages()

for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: What is this page about? https://en.wikipedia.org/wiki/Capybara

response
TOOL CALL: get_page {"page_title":"Capybara"}

request
TOOL RETURN: get_page

response
The Wikipedia page for the **capybara** discusses various aspects of this animal, which is recognized as the largest living rodent. Here are key points from the page:

1. **Species Information**: The capybara (scientific name *Hydrochoerus hydrochaeris*) is native to South America, where it inhabits savannas and forests, often near bodies of water. 

2. **Physical Description**: Capybaras have a distinct barrel-shaped body, reddish-brown fur, and they typically weigh between 35 to 66 kg (about 77 to 146 lbs).

3. **Social Structure**: They are highly social animals, often found in groups ranging from 10 to 20 individuals, but can congregate in larger groups during certain seasons.

4. **Diet**: Capybaras are herbivores, primarily feeding on grasses and aquatic plants. They demonstrate unique feeding behavior, incl

## 6. Testing Your Agent - Search Then Fetch

In [82]:
query = "What are the main threats to capybara populations?"
result = await search_agent.run(query)

In [83]:
messages = result.all_messages()

for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: What are the main threats to capybara populations?

response
TOOL CALL: search_wikipedia {"query":"Capybara"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: get_page {"page_title":"Capybara"}

request
TOOL RETURN: get_page

response
The main threats to capybara populations include:

1. **Hunting**: Capybaras are often hunted for their meat, pelts, and fat, which has led to decreases in local populations in some areas.

2. **Human-Wildlife Conflict**: In agricultural areas, capybaras may be seen as competition for livestock grazing land, leading to them being killed by farmers.

3. **Habitat Loss**: Urbanization, agriculture, and deforestation can alter or destroy the wetlands where capybaras live, negatively impacting their populations.

4. **Predation**: While not a direct threat from humans, natural predators such as jaguars and caimans can impact capybara numbers, particularly among the young and vulnerable individuals.

Despite these threats, the overall 